# DiffKV Private Colab 7B GPU Benchmark Runner
Run this notebook top-to-bottom on a Colab T4 GPU (or better) runtime to securely clone your private repository, install dependencies, enable NF4 quantization, and run benchmarks for the active runtime.

## Step 1: Securely Clone Private Repository
Paste your fine-grained GitHub PAT when prompted. This code downloads the zip directly and deletes it after extraction, leaving no trace of the credentials in `.git/config`.

In [ ]:
import os
import shutil
import getpass
import urllib.request
import zipfile

# Clean the old directory if it exists
if os.path.exists('/content/Differential-KV'):
    print("Deleting old directory...")
    shutil.rmtree('/content/Differential-KV')

# Move to /content
os.chdir('/content')

# Ask for token and clone
token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")
url = "https://api.github.com/repos/Omc12/Differential-KV/zipball"

req = urllib.request.Request(url)
req.add_header('Authorization', f'token {token}')
req.add_header('User-Agent', 'python-urllib')

print("Downloading clean repository zip archive from GitHub...")
try:
    with urllib.request.urlopen(req) as response:
        with open('repo_temp.zip', 'wb') as f:
            f.write(response.read())
            
    print("Extracting...")
    with zipfile.ZipFile('repo_temp.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    
    extracted_dirs = [d for d in os.listdir('.') if os.path.isdir(d) and 'Differential-KV' in d]
    if extracted_dirs:
        os.rename(extracted_dirs[0], 'Differential-KV')
        os.remove('repo_temp.zip')
        print("\nSuccess! Clean repository cloned.")
        os.chdir('Differential-KV')
        print(f"Current working directory: {os.getcwd()}")
    else:
        print("Could not locate extracted folder.")
except Exception as e:
    print(f"Error: {e}")

## Step 2: Install Dependencies
Includes `bitsandbytes` to enable 4-bit NF4 quantization on CUDA.

In [ ]:
!pip install -q triton psutil tabulate transformers accelerate pytest optimum bitsandbytes

## Step 3: Run Custom Triton Combined Kernel Unit Tests

In [ ]:
!python -m pytest ACTIVE_RUNTIME/tests/test_triton_combined.py -v

## Step 4: Run Main Paper Benchmark (Qwen2.5-7B-Instruct)
We export `DIFFKV_QUANTIZATION=nf4` so the 7B model fits comfortably on the 15/16GB VRAM of a T4 GPU.

In [ ]:
%env DIFFKV_QUANTIZATION=nf4
!python paper/scripts/measure_active.py --ctx 4096 8192 --modes compressed exact --gen 64 --out paper_results.json

## Step 5: Display Results

In [ ]:
import json
try:
    with open("paper_results.json") as f:
        data = json.load(f)
    print(json.dumps(data, indent=2))
except FileNotFoundError:
    print("Results file not found. Make sure Step 4 completed successfully.")